In [ ]:
import lightning.pytorch as pl
import pandas as pd

from sportsml.cbb.data.features import (
    DATE_COLUMN,
    GRAPH_STATS_COLUMNS,
    SEASON_COLUMN,
    TARGET_COLUMN,
    TEAM_COLUMN,
    TEAM_OPP_COLUMN,
)
from sportsml.graph.datamodule import GraphDataModule
from sportsml.graph.model import GraphModel
from sportsml.graph.nn.encoder.edge_conv_encoder import EdgeConvEncoder
from sportsml.graph.nn.predictor.ffn import EdgeFFN
from sportsml.graph.nn.predictor.inner_product import InnerProductPredictor


In [ ]:
games = pd.read_csv('../data/cbb/data.csv')

In [ ]:
dm = GraphDataModule(
    games=games[games[SEASON_COLUMN] >= 2016],
    stats_columns=GRAPH_STATS_COLUMNS,
    target_column=TARGET_COLUMN,
    season_column=SEASON_COLUMN,
    date_column=DATE_COLUMN,
    batch_size=16,
)

In [ ]:
encoder = EdgeConvEncoder(
    edge_dim=len(GRAPH_STATS_COLUMNS),
    hidden_dim=300,
    out_dim=100,
    num_layers=3,
)

predictor = EdgeFFN(
    in_dim=100,
    out_dim=1,
    hidden_dim=100,
)

# predictor = InnerProductPredictor()

model = GraphModel(
    encoder=encoder,
    predictor=predictor,
    target_mean=dm.target_mean,
    target_std=dm.target_std,
)

In [ ]:
trainer = pl.Trainer(
    max_epochs=100,
    logger=pl.loggers.WandbLogger(project="sportsml-cbb", log_model=True),
    callbacks=[
        pl.callbacks.ModelCheckpoint(monitor="val_loss", mode="min"),
        pl.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=10),
    ],
)

In [ ]:
trainer.fit(model, dm)

In [ ]:
trainer.test(model, dm, ckpt_path="best", weights_only=False)

In [ ]:
best = GraphModel.load_from_checkpoint(trainer.checkpoint_callback.best_model_path)

In [ ]:
from sportsml.graph.model import SportsMLPredictor

In [ ]:
from sportsml.cbb.data.nodes import team_name_map

In [ ]:
g = dm.get_latest_graph()
x = best.encoder(edge_index=g.edge_index, edge_attr=g.edge_attr)

In [ ]:
mlflow_predictor = SportsMLPredictor(
    model=best,
    team_embeddings=x)

In [ ]:
import mlflow

In [ ]:
mlflow.pyfunc.save_model("../models/cbb/pyg_edge_conv", python_model=mlflow_predictor)

In [ ]:
mlflow.pyfunc.log_model(python_model=mlflow_predictor)